# Orchestration contextuelle des experts — validation

A = stabilité, B = structure P1_G1, C = régime seulement si admissible. **Aucun forecast 2026**, aucun tuning, A/B/D et fondation inchangés. Deux moteurs de confiance séparés : occurrence et croissance → contributions unitaires en mémoire → médiane P1. Toutes les sorties sont agrégées.

[Protocole et rapport complet](../docs/contextual_orchestration.md). La sécurité à cutoff conserve la réserve de fidélité historique CRM.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if not (ROOT/'src').is_dir(): ROOT=ROOT.parent
assert (ROOT/'src').is_dir(), 'Exécuter depuis le dépôt ou notebooks/.'
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.models.contextual_experts import RULES, rules_fingerprint
from src.evaluation.contextual_orchestration import audit_contextual_signals, run_contextual_orchestration
leases=pd.read_csv(ROOT/'data/raw/equinoxe_lease_history.csv',dtype={'sPropCode':'string','sUnitCode':'string'})
external=pd.read_csv(ROOT/'data/external/external_signals.csv')
# Aucun bail ou vecteur unitaire n’est affiché/exporté.


## Audit à cutoff et choix de C, avant scoring

C candidat unique : dernière cohorte historique calendaire Y−1, moyenne O qualifiés pour occurrence, médiane G des unités positives pour croissance. Support minimum 30 et couverture des labels ≥80 %. Croissances négatives conservées. Aucun lookback sélectionné par MAE.

Les deux C sont **NONE** aux trois cutoffs : 10/350, 8/405, 13/645 labels connus, 5/4/2 positifs. Les gaps, concessions/asking et externes non qualifiés restent exclus ; les autres signaux sont audités sans tournoi de C.

In [2]:
signal_audit=audit_contextual_signals(leases,external)
display(signal_audit)
selected=signal_audit.loc[signal_audit.signal.str.startswith('C_')]
assert selected.status.eq('NONE').all()
display(selected[['year','component','support','coverage','estimate_pct','status']])

,year,component,signal,definition,support,coverage,estimate_pct,status,reason,availability
0,2023,growth,C_growth : dernière cohorte historique,Médiane G des unités positives de Y−1 mûres à D.,5.0,0.028571,1.130435,NONE,Support >=30 et couverture labels >=80% requis...,SAFE_WITH_CONDITIONS
1,2023,growth,Croissance récente par transition,Médiane effective annualisée des transitions d...,7.0,0.019499,2.264068,AUDIT_ONLY,Maturité sélective ; population différente des...,SAFE_WITH_CONDITIONS
2,2023,growth,Médiane glissante trois cohortes,Médiane G positifs des cohortes Y−3 à Y−1 qual...,647.0,0.653162,3.323608,AUDIT_ONLY,Retard de maturité ; signal plus ancien et pro...,SAFE_WITH_CONDITIONS
3,2023,growth,Divergence récent/long terme,"C_growth moins médiane historique de A, si C q...",5.0,0.028571,NaN,CONTEXT_ONLY,"Confiance de C, jamais expert indépendant supp...",SAFE_WITH_CONDITIONS
4,2023,growth,Momentum renouvellement,Croissance des transitions historiques mûres d...,5.0,0.013928,1.130435,AUDIT_ONLY,"Faible historique récent ; type futur inconnu,...",SAFE_WITH_CONDITIONS
...,...,...,...,...,...,...,...,...,...,...
61,2025,occurrence,Concentration échéances,Caractéristiques du dernier bail connu et comp...,856.0,1.000000,NaN,CONTEXT_ONLY,"Contexte reconstructible, mais aucune relation...",SAFE_WITH_CONDITIONS
62,2025,occurrence,Changement de mix candidat,Caractéristiques du dernier bail connu et comp...,856.0,1.000000,NaN,CONTEXT_ONLY,"Contexte reconstructible, mais aucune relation...",SAFE_WITH_CONDITIONS
63,2025,occurrence,HHI bâtiment,Caractéristiques du dernier bail connu et comp...,856.0,1.000000,NaN,CONTEXT_ONLY,"Contexte reconstructible, mais aucune relation...",SAFE_WITH_CONDITIONS
64,2025,growth,Régime externe/réglementaire,Filtre public existant : date vérifiée <=D et ...,0.0,NaN,NaN,REJECT_EXTERNAL_FOR_C,Aucun signal historiquement qualifié et utile ...,UNKNOWN


,year,component,support,coverage,estimate_pct,status
0,2023,growth,5.0,0.028571,1.130435,NONE
10,2023,occurrence,10.0,0.028571,50.000000,NONE
22,2024,growth,4.0,0.019753,NaN,NONE
32,2024,occurrence,8.0,0.019753,50.000000,NONE
44,2025,growth,2.0,0.020155,NaN,NONE
54,2025,occurrence,13.0,0.020155,15.384615,NONE


## Formules et freeze avant backtests

`t_A=1`. `q_B=n_B/(n_B+30)` au dernier niveau effectivement appris, nul si repli global. `d_B=min(|B−A|,|B−global_B|)` ; `t_B=q_B*d_B/(d_B+S)`.

`S_p=max(sqrt(p_A*(1−p_A)),1/(n_A+1))`, `S_g=max(IQR(G_A),|g_A|,epsilon_machine)`.

C qualifié seulement si n≥30, couverture≥0,8, estimation/dispersion finies. `t_C=qualification*n/(n+30)*coverage*S/(S+dispersion_C)*|C−A|/(|C−A|+S)`. Sinon 0.

`w_E=t_E/somme(t)` séparément pour p et g. A reste l'ancre, B/C manquants ne cassent pas le forecast. HHI et composition sont diagnostics, sans règle inventée à partir des absences futures. Constantes fixées sur principes de support, pas sur les erreurs annuelles.

In [3]:
FROZEN_FINGERPRINT='f8ce4bd1355a1ab3ee238bdbbb8754e38a70d332b72f5548c1187952dc51fa07'
assert rules_fingerprint()==FROZEN_FINGERPRINT, 'Protocole modifié : arrêter avant scoring.'
print('Règles immuables :',RULES)
print('Empreinte :',FROZEN_FINGERPRINT)
result=run_contextual_orchestration(leases,external,expected_fingerprint=FROZEN_FINGERPRINT)
assert result['rules_fingerprint']==FROZEN_FINGERPRINT
for name,value in result.items():
    if isinstance(value,pd.DataFrame):
        assert not {'UNIT_KEY','sUnitCode','sPropCode','hTenant','hLease'}.intersection(value.columns),name
display(result['checks'])

Règles immuables : TrustRules(support_scale=30, recent_minimum=30, recent_coverage_minimum=0.8, recent_window='previous_calendar_year', anchor_trust=1.0)
Empreinte : f8ce4bd1355a1ab3ee238bdbbb8754e38a70d332b72f5548c1187952dc51fa07


,year,prediction_origin,candidate_units,observed_units,train_positive_unit_years,alpha,identical_A_occurrence,identical_A_cohort_and_target,g_global_pct,temporal_status
0,2023,2022-12-31,405,401,882,30.0,True,True,2.918870,SAFE_WITH_CONDITIONS
1,2024,2023-12-31,645,634,1227,30.0,True,True,3.319802,SAFE_WITH_CONDITIONS
2,2025,2024-12-31,856,773,1627,30.0,True,True,3.024079,SAFE_WITH_CONDITIONS


## Backtests et comparaison des cinq références

P1 en %, erreurs en pp. A, B full, 50/50 et D sont reproduits, sans nouvelle sélection ni modification. D reste rejeté. Les vecteurs/poids orchestrés ont tous été fixés avant révélation des labels.

In [4]:
display(result['results'].round(6))
display(result['metrics'].round(6))

,year,model,prediction_pct,realized_pct,error_pp,absolute_error_pp
0,2023,A,2.870060,2.036102,0.833957,0.833957
1,2023,Model B — full hierarchical,2.871314,2.036102,0.835212,0.835212
2,2023,A/B 50-50,2.870687,2.036102,0.834585,0.834585
3,2023,D,2.870927,2.036102,0.834825,0.834825
4,2024,A,3.269179,1.607852,1.661326,1.661326
5,2024,Model B — full hierarchical,3.236099,1.607852,1.628247,1.628247
6,2024,A/B 50-50,3.252639,1.607852,1.644787,1.644787
7,2024,D,3.289002,1.607852,1.681150,1.681150
8,2025,A,2.969328,3.300878,-0.331551,0.331551
9,2025,Model B — full hierarchical,2.924144,3.300878,-0.376735,0.376735


,model,MAE_pp,RMSE_pp,bias_pp,worst_absolute_error_pp
0,A,0.942278,1.090171,0.721244,1.661326
1,Model B — full hierarchical,0.946731,1.078687,0.695575,1.628247
2,A/B 50-50,0.944505,1.084323,0.708410,1.644787
3,D,0.949175,1.100473,0.728141,1.681150
4,ORCHESTRATED,0.942810,1.091772,0.724022,1.663993


## Experts, poids et diagnostics annuels

Taux moyens sur toute la cohorte ; médianes conditionnelles sur les mêmes unités positives pour le scoring seulement. Les poids varient par candidate : leur moyenne n'est pas un poids appliqué aux moyennes d'experts. Aucun résultat réalisé n'entre dans le contexte de confiance.

In [5]:
display(result['annual'].round(6))
display(result['building_weights'].round(6))

,year,cohort_size,realized_transition_pct,realized_conditional_growth_pct,predicted_P1_pct,realized_P1_pct,error_pp,absolute_error_pp,occurrence_error_pp,Brier,...,C_growth_pct,weight_C_g,activation_C_g,A_dominance_g,B_global_fallback_share_g,B_support_share_g,B_median_support_g,B_deviation_from_global_pp_g,hierarchical_child_support_share_p,B_parent_fallback_share_p
0,2023,405,99.012346,2.085372,2.872357,2.036102,0.836255,0.836255,-0.814723,0.009294,...,NaN,0.0,0.0,1.0,0.000000,1.000000,944.0,0.012651,1.000000,0.0
1,2024,645,98.294574,1.699619,3.271845,1.607852,1.663993,1.663993,0.015403,0.016215,...,NaN,0.0,0.0,1.0,0.148837,0.851163,1308.0,0.009453,0.851163,0.0
2,2025,856,90.303738,3.716980,2.972696,3.300878,-0.328182,0.328182,7.577372,0.092567,...,NaN,0.0,0.0,1.0,0.140187,0.859813,1740.0,0.014550,0.859813,0.0


,year,building,component,candidate_units,cohort_share_pct,B_support_mean,B_global_fallback_share,C_selected,recent_support,recent_coverage,weight_A,weight_B,weight_C,final_component_pct
0,2023,Daniel-Johnson,occurrence,111,27.407407,84.765766,0.0,False,10,0.028571,0.932489,0.067511,0.0,98.366768
1,2023,Levesque,occurrence,81,20.000000,88.481481,0.0,False,10,0.028571,0.918928,0.081072,0.0,98.147926
2,2023,Saint-Elzear,occurrence,213,52.592593,86.906103,0.0,False,10,0.028571,0.919413,0.080587,0.0,98.128375
3,2023,Daniel-Johnson,growth,111,27.407407,944.000000,0.0,False,5,0.028571,0.997551,0.002449,0.0,2.918779
4,2023,Levesque,growth,81,20.000000,944.000000,0.0,False,5,0.028571,0.997551,0.002449,0.0,2.918779
5,2023,Saint-Elzear,growth,213,52.592593,944.000000,0.0,False,5,0.028571,0.997551,0.002449,0.0,2.918779
6,2024,Le Carlyle,occurrence,127,19.689922,124.677165,0.0,False,8,0.019753,0.914853,0.085147,0.0,98.370798
7,2024,Daniel-Johnson,occurrence,125,19.379845,112.808000,0.0,False,8,0.019753,0.916978,0.083022,0.0,98.340488
8,2024,Levesque,occurrence,82,12.713178,120.609756,0.0,False,8,0.019753,0.905667,0.094333,0.0,98.078302
9,2024,The Met,occurrence,96,14.883721,0.000000,1.0,False,8,0.019753,1.000000,0.000000,0.0,98.475120


## Conclusions 2023 / 2024 / 2025

**2023** : poids occurrence A/B 92,290/7,710 %, croissance 99,755/0,245 %, C=0. P1 2,872357 %, réalisé 2,036102 %, erreur +0,836255 pp. Le Brier diminue, mais la croissance reste surestimée.

**2024 — NO RELIABLE POINT-IN-TIME REGIME SIGNAL FOUND.** 8/405 labels récents, 4 positifs : C=0. Poids occurrence A/B 92,645/7,355 %, croissance 99,831/0,169 %. Croissance finale 3,319653 %, réalisée 1,699619 %. P1 3,271845 %, erreur +1,663993 pp. Le système refuse de surinterpréter un historique immature ; il ne détecte pas le ralentissement manquant.

**2025** : 13/645 labels récents, 2 positifs : C=0. Poids occurrence A/B 91,417/8,583 %, croissance 99,744/0,256 %. À Saint-Elzear, taux final 97,974 %, réalisé 69,907 % : aucune anticipation fiable de la rupture démontrée. P1 2,972696 %, erreur −0,328182 pp. Les absences 2025 servent uniquement au diagnostic après scoring.

MAE orchestrée **0,942810 pp** contre A 0,942278 ; pire erreur **1,663993** contre 1,661326. Aucun gain P1 structurel démontré. Le Brier d'occurrence est légèrement meilleur sur chaque année. Trois années déjà examinées ne sont pas un holdout indépendant.

## Décision et arrêt

**ORCHESTRATION_READY_TO_FREEZE**, sous les hypothèses de fidélité CRM de la fondation : architecture explicable, règles déterministes, experts spécialisés seulement si supportés, fallback A robuste et poids séparés. Cela ne certifie pas un gain futur ni des versions historiques archivées. C reste absent ici.

Revue humaine avant 2026. **Aucun forecast 2026**, aucun tuning supplémentaire, aucun système/frontend ajouté, aucun commit ou push.